In [42]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [43]:
!nvidia-smi

Mon Aug 24 06:41:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [44]:
%%writefile san.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloFromGpu() {
    printf("Hello World from GPU!\n");
}

int main() {
    printf("Hello World from CPU!\n");

    helloFromGpu<<<1, 10>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing san.cu


In [45]:
!nvcc san.cu -o san


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [46]:
!./san

Hello World from CPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!
Hello World from GPU!


In [47]:
%%writefile vec.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void vecAdd(int *a, int *b, int *c);

int main() {

    // Host arrays (CPU memory)
    int h_a[N] = {1, 2, 3, 4, 5, 6, 7, 8};
    int h_b[N] = {10, 20, 30, 40, 50};
    int h_c[N];

    // Device pointers (GPU memory)
    int *d_a, *d_b, *d_c;

    // Allocate GPU memory
    cudaMalloc((void**)&d_a, N * sizeof(int));
    cudaMalloc((void**)&d_b, N * sizeof(int));
    cudaMalloc((void**)&d_c, N * sizeof(int));

    // Copy data from CPU to GPU
    cudaMemcpy(d_a, h_a, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N * sizeof(int), cudaMemcpyHostToDevice);

    // Launch kernel
    vecAdd<<<1, N>>>(d_a, d_b, d_c);

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    // Copy result from GPU to CPU
    cudaMemcpy(h_c, d_c, N * sizeof(int), cudaMemcpyDeviceToHost);

    // Print result
    printf("Result:\n");

    for (int i = 0; i < N; i++) {
        printf("%d ", h_c[i]);
    }

    printf("\n");

    // Free GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

// GPU kernel
__global__ void vecAdd(int *a, int *b, int *c)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        c[idx] = a[idx] + b[idx];
    }
}

Writing vec.cu


In [48]:
!nvcc vec.cu -o vec

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [49]:
!./vec

Result:
11 22 33 44 55 6 7 8 


In [50]:
%%writefile red.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 1024
#define BLOCK_SIZE 256

// Kernel for parallel reduction
__global__ void reduceSum(int *input, int *output)
{
    __shared__ int sharedData[BLOCK_SIZE];

    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Load data into shared memory
    sharedData[tid] = (i < N) ? input[i] : 0;

    __syncthreads();

    // Parallel reduction
    for (int stride = blockDim.x / 2; stride > 0; stride /= 2)
    {
        if (tid < stride)
        {
            sharedData[tid] += sharedData[tid + stride];
        }

        __syncthreads();
    }

    // Store result of this block
    if (tid == 0)
    {
        output[blockIdx.x] = sharedData[0];
    }
}

int main()
{
    int h_input[N];
    int h_output;
    
    int *d_input, *d_output;

    // Initialize input
    for (int i = 0; i < N; i++)
    {
        h_input[i] = 1;
    }

    int numBlocks = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;

    // Allocate device memory
    cudaMalloc((void **)&d_input, N * sizeof(int));
    cudaMalloc((void **)&d_output, numBlocks * sizeof(int));

    // Copy input to GPU
    cudaMemcpy(d_input, h_input, N * sizeof(int),
               cudaMemcpyHostToDevice);

    // Launch kernel
    reduceSum<<<numBlocks, BLOCK_SIZE>>>(d_input, d_output);

    // Copy partial sums back to CPU
    int *h_partial = (int *)malloc(numBlocks * sizeof(int));

    cudaMemcpy(h_partial, d_output, numBlocks * sizeof(int),
               cudaMemcpyDeviceToHost);

    // Final reduction on CPU
    h_output = 0;

    for (int i = 0; i < numBlocks; i++)
    {
        h_output += h_partial[i];
    }

    printf("Sum = %d\n", h_output);

    // Free memory
    free(h_partial);
    cudaFree(d_input);
    cudaFree(d_output);

    return 0;
}

Overwriting red.cu


In [51]:
!nvcc red.cu -o red

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
^C

Compilation terminated.
nvcc error   : 'cudafe++' died due to signal 2 


In [52]:
!./red

Sum = 1024


In [53]:
%%writefile exc_scan.cu
#include<stdio.h>
#include<cuda_runtime.h>
#define N 8 

__global__ void exclusiveScan(int *a , int *b);
int main(){
    int h_a[N] = {1,2,3,4,5,6,7,8};
    int h_b[N];
    int *d_a,*d_b;
    cudaMalloc((void**)&d_a,N*sizeof(int));
    cudaMalloc((void**)&d_b,N*sizeof(int));
    cudaMemcpy(d_a,h_a,N*sizeof(int),cudaMemcpyHostToDevice);
    exclusiveScan<<<1,N>>>(d_a,d_b);
    cudaDeviceSynchronize();
    cudaMemcpy(h_b,d_b,N*sizeof(int),cudaMemcpyDeviceToHost);
    printf("Exclusive Scan Sum:\n");
    for(int i=0 ; i<N ; i++){
        printf("%d ", h_b[i]);
    }
    printf("\n");
    cudaFree(d_a);
    cudaFree(d_b);
    return 0;
}
__global__ void exclusiveScan(int *a, int *b){
    int idx = threadIdx.x;
    if(idx<N){
        int sum = 0;
        for(int i = 0; i<idx ;i++){
            sum += a[i];
        }
        b[idx] = sum;
    }
    

}

Overwriting exc_scan.cu


In [54]:
!nvcc exc_scan.cu -o exc_scan

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
^C


In [55]:
!./exc_scan

Exclusive Scan Sum:
0 1 3 6 10 15 21 28 


In [56]:
%%writefile inc_scan.cu
#include<stdio.h>
#include<cuda_runtime.h>
#define N 8 

__global__ void inclusiveScan(int *a , int *b);
int main(){
    int h_a[N] = {1,2,3,4,5,6,7,8};
    int h_b[N];
    int *d_a,*d_b;
    cudaMalloc((void**)&d_a,N*sizeof(int));
    cudaMalloc((void**)&d_b,N*sizeof(int));
    cudaMemcpy(d_a,h_a,N*sizeof(int),cudaMemcpyHostToDevice);
    inclusiveScan<<<1,N>>>(d_a,d_b);
    cudaDeviceSynchronize();
    cudaMemcpy(h_b,d_b,N*sizeof(int),cudaMemcpyDeviceToHost);
    printf("inclusive Scan Sum:\n");
    for(int i=0 ; i<N ; i++){
        printf("%d ", h_b[i]);
    }
    printf("\n");
    cudaFree(d_a);
    cudaFree(d_b);
    return 0;
}
__global__ void inclusiveScan(int *a, int *b){
    int idx = threadIdx.x;
    if(idx<N){
        int sum = 0;
        for(int i = 0; i<=idx ;i++){
            sum += a[i];
        }
        b[idx] = sum;
    }
    

}

Overwriting inc_scan.cu


In [57]:
!nvcc inc_scan.cu -o inc_scan

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [58]:
!./inc_scan

inclusive Scan Sum:
1 3 6 10 15 21 28 36 


In [59]:
%%writefile both.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void scan(int *a, int *b, int choice)
{
    int idx = threadIdx.x;

    if (idx < N)
    {
        int sum = 0;

        if (choice == 1)   // Exclusive Scan
        {
            for (int i = 0; i < idx; i++)
            {
                sum += a[i];
            }
        }
        else if (choice == 2)   // Inclusive Scan
        {
            for (int i = 0; i <= idx; i++)
            {
                sum += a[i];
            }
        }

        b[idx] = sum;
    }
}

int main()
{
    int h_a[N] = {1, 2, 3, 4, 5, 6, 7, 8};
    int h_b[N];

    int *d_a, *d_b;
    int choice;

    printf("Enter your choice:\n");
    printf("1. Exclusive Scan\n");
    printf("2. Inclusive Scan\n");
    scanf("%d", &choice);

    if (choice != 1 && choice != 2)
    {
        printf("Invalid choice!\n");
        return 0;
    }

    cudaMalloc((void**)&d_a, N * sizeof(int));
    cudaMalloc((void**)&d_b, N * sizeof(int));

    cudaMemcpy(d_a, h_a, N * sizeof(int),
               cudaMemcpyHostToDevice);

    scan<<<1, N>>>(d_a, d_b, choice);

    cudaDeviceSynchronize();

    cudaMemcpy(h_b, d_b, N * sizeof(int),
               cudaMemcpyDeviceToHost);

    if (choice == 1)
        printf("Exclusive Scan Sum:\n");
    else
        printf("Inclusive Scan Sum:\n");

    for (int i = 0; i < N; i++)
    {
        printf("%d ", h_b[i]);
    }

    printf("\n");

    cudaFree(d_a);
    cudaFree(d_b);

    return 0;
}

Overwriting both.cu


In [60]:
!nvcc both.cu -o both

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
^C
nvcc error   : 'gcc' died due to signal 2 


In [65]:
!echo 1 | ./both

Enter your choice:
1. Exclusive Scan
2. Inclusive Scan
Exclusive Scan Sum:
0 1 3 6 10 15 21 28 


In [66]:
!echo 2 | ./both

Enter your choice:
1. Exclusive Scan
2. Inclusive Scan
Inclusive Scan Sum:
1 3 6 10 15 21 28 36 


In [67]:
!echo 3 | ./both

Enter your choice:
1. Exclusive Scan
2. Inclusive Scan
Invalid choice!


In [13]:
%%writefile list.cu
#include<stdio.h>
#include<cuda_runtime.h>
#define N 3
__global__ void listRanking(int *next,int *rank);
int main(){
    int h_next[N] = {1,2,-1};
    int h_rank[N];
    int *d_next , *d_rank;
    printf("___________________\n");
    printf("List rank using Cuda:\n");
    printf("____________________\n");
    printf("Linked List:\n");
    for(int i=0;i<N;i++){
        if(h_next[i] != -1){
            printf("%d -> %d\n",i,h_next[i]);
        }
        else{
            printf("%d -> NULL\n", i);
        }
        
    }
    printf("\nAllocating GPU  Memory.......\n");
    cudaMalloc((void**)&d_next,N*sizeof(int));
    cudaMalloc((void**)&d_rank,N*sizeof(int));
    cudaMemcpy(d_next,h_next,N*sizeof(int),cudaMemcpyHostToDevice);
    listRanking<<<1,N>>>(d_next,d_rank);
    cudaDeviceSynchronize();
    printf("Kernel execution completed:\n");
    cudaMemcpy(h_rank,d_rank,N*sizeof(int),cudaMemcpyDeviceToHost);
    printf("Result copied Successfullyy \n");
    printf("\n Final Node Ranks\n");
    printf("___________\n");
    for(int i=0;i<N;i++){
        printf("Node %d : Rank = %d\n",i,h_rank[i]);
    }
    printf("Releasing Gpu Memory...\n");
    cudaFree(d_next);
    cudaFree(d_rank);
    return 0;
 
}
__global__ void listRanking(int *next,int *rank){
    int idx = threadIdx.x;
    if(idx<N){
        int current = 0;
        int r = 0;
        printf("Thread %d started\n",idx);
        while(current != idx){
            printf("Thread %d visiting Node %d\n",idx,current);
            current = next[current];
            r++;
        }
        printf("Thread %d: Reached Node %d\n",idx,idx);
        rank[idx] = r;
        printf("Thread %d : Stored Rank = %d\n\n",idx,r);
    }
}

Overwriting list.cu


In [14]:
!nvcc list.cu -o list

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [15]:
!./list

___________________
List rank using Cuda:
____________________
Linked List:
0 -> 1
1 -> 2
2 -> NULL

Allocating GPU  Memory.......
Thread 0 started
Thread 1 started
Thread 2 started
Thread 1 visiting Node 0
Thread 2 visiting Node 0
Thread 2 visiting Node 1
Thread 0: Reached Node 0
Thread 1: Reached Node 1
Thread 2: Reached Node 2
Thread 0 : Stored Rank = 0

Thread 1 : Stored Rank = 1

Thread 2 : Stored Rank = 2

Kernel execution completed:
Result copied Successfullyy 

 Final Node Ranks
___________
Node 0 : Rank = 0
Node 1 : Rank = 1
Node 2 : Rank = 2
Releasing Gpu Memory...
